# 32c. Gateway Routing & Unit Economics

**Tier:** Production & Safety
**Estimated time:** 55 minutes
**Prerequisites:** 24 (LLM evals fundamentals), 32 (cost engineering)
**Priority:** 🟡 Important — the point where "our AI feature costs money" becomes "our AI feature has a P&L." *If skipped, revisit when:* someone asks "what does this feature actually cost per user," or you're choosing between LiteLLM, a hand-rolled router, and just picking one model and hoping.
**Source material:** Notebook 32's cost model (routing, caching, budgets); LiteLLM's cost-based/latency-based routing strategies; current Anthropic and OpenAI published API pricing (verified live, not recalled).

## What You'll Learn
- How to model **unit economics** for an AI feature: cost per request, and — the number that actually decides whether a flat-price plan survives — the *distribution* of cost per user
- Why notebook 32's "cheap-first, escalate-on-hedge" router is a starting point, not the whole picture: a real router needs a **scorecard** — measured latency, real cost, and a quality score from an actual eval — not a single heuristic
- How to build that scorecard-based router from scratch, and route each request against *declared constraints* (a quality floor, a latency ceiling, a budget)
- Where the off-the-shelf version of all this lives: **LiteLLM** as a routing gateway, and **LM Studio**/**Ollama** as local OpenAI-compatible backends a router can treat as just another tier

## Why This Matters
Notebook 32 taught you the individual levers — routing, caching, batching, budgets. This notebook is what happens when those levers meet a real business model: a subscription price, a population of users with wildly different usage, and a router that has to make a real decision (not just "cheap first") for every single request. Get the economics wrong and a feature that looked profitable in a demo bleeds money at scale; get the routing wrong and you're either overpaying for easy requests or failing hard ones.


## Part 1 — Unit economics: the number that matters is the *distribution*, not the average

A flat-price plan ("$20/month, unlimited") is a bet that average usage stays comfortably under the price. The bet fails at the *tail*: a handful of power users can burn far more in tokens than the plan charges, and if enough of them show up, the aggregate average can still look fine while a real, growing subset of your user base is actively loss-making.


In [1]:
import numpy as np

rng = np.random.default_rng(0)

# Anthropic's current published per-million-token pricing (verified via the claude-api
# skill's live model table, not recalled) -- this is what "cost per request" actually means.
PRICING = {
    "haiku":  {"input": 1.00, "output": 5.00},    # claude-haiku-4-5
    "sonnet": {"input": 3.00, "output": 15.00},   # claude-sonnet-5
    "opus":   {"input": 5.00, "output": 25.00},   # claude-opus-5
}

def cost_per_request(tier, in_tok, out_tok):
    p = PRICING[tier]
    return (in_tok / 1e6) * p["input"] + (out_tok / 1e6) * p["output"]

# A population of 2,000 subscribers on a $20/month plan, using a mid-tier model.
# Monthly request COUNT is heavy-tailed -- most people use the feature occasionally,
# a minority use it constantly (a genuinely common shape for usage-based products).
N_USERS = 2000
SUBSCRIPTION_PRICE = 20.0
AVG_IN_TOK, AVG_OUT_TOK = 800, 500   # a realistic per-request size, not a toy example

monthly_requests = rng.lognormal(mean=4.0, sigma=1.8, size=N_USERS)
cost_per_user = monthly_requests * cost_per_request("sonnet", AVG_IN_TOK, AVG_OUT_TOK)
margin_per_user = SUBSCRIPTION_PRICE - cost_per_user

print(f"Median monthly requests: {np.median(monthly_requests):.0f}   "
      f"P99: {np.percentile(monthly_requests, 99):.0f}")
print(f"Median cost/user: ${np.median(cost_per_user):.2f}   "
      f"P99: ${np.percentile(cost_per_user, 99):.2f}   "
      f"P99.5: ${np.percentile(cost_per_user, 99.5):.2f}")
print(f"Average margin per user: ${margin_per_user.mean():.2f}  (looks healthy)")
print(f"Fraction of users who are ALREADY loss-making (cost > $20): {np.mean(cost_per_user > 20):.2%}")
print(f"Fraction burning past $40 (2x the plan price): {np.mean(cost_per_user > 40):.2%}")


Median monthly requests: 51   P99: 3164
Median cost/user: $0.51   P99: $31.32   P99.5: $46.31
Average margin per user: $17.63  (looks healthy)
Fraction of users who are ALREADY loss-making (cost > $20): 1.85%
Fraction burning past $40 (2x the plan price): 0.65%


The average margin looks comfortable — but that average is computed over a population where roughly 1 in 50 users is *already* costing more than they pay, and a smaller group is burning double the subscription price or more. That's the canonical failure mode: **a healthy-looking average hides a real and growing loss-making tail**, and the tail is exactly the group a genuinely popular product creates more of over time, not fewer.

## Sensitivity: does caching or routing help more?


In [2]:
def population_margin(cache_hit_rate=0.0, cheap_fraction=0.0):
    """cache_hit_rate: fraction of INPUT tokens served from a prompt cache (~90% cheaper).
    cheap_fraction: fraction of REQUESTS routed to the haiku tier instead of sonnet.
    """
    effective_in_price = PRICING["sonnet"]["input"] * (1 - cache_hit_rate * 0.9)
    blended_in = effective_in_price * (1 - cheap_fraction) + PRICING["haiku"]["input"] * cheap_fraction
    blended_out = PRICING["sonnet"]["output"] * (1 - cheap_fraction) + PRICING["haiku"]["output"] * cheap_fraction
    cost = monthly_requests * ((AVG_IN_TOK / 1e6) * blended_in + (AVG_OUT_TOK / 1e6) * blended_out)
    return (SUBSCRIPTION_PRICE - cost).mean(), np.mean(cost > SUBSCRIPTION_PRICE)

print("Cache hit-rate sensitivity (no routing change):")
for cache in [0.0, 0.3, 0.6, 0.9]:
    margin, neg_frac = population_margin(cache_hit_rate=cache)
    print(f"  cache_hit_rate={cache:.1f}  avg margin=${margin:.2f}  loss-making users={neg_frac:.2%}")

print("\nRouting-mix sensitivity (no caching change):")
for cheap in [0.0, 0.3, 0.6, 0.9]:
    margin, neg_frac = population_margin(cheap_fraction=cheap)
    print(f"  cheap_routing_fraction={cheap:.1f}  avg margin=${margin:.2f}  loss-making users={neg_frac:.2%}")


Cache hit-rate sensitivity (no routing change):
  cache_hit_rate=0.0  avg margin=$17.63  loss-making users=1.85%
  cache_hit_rate=0.3  avg margin=$17.78  loss-making users=1.70%
  cache_hit_rate=0.6  avg margin=$17.94  loss-making users=1.65%
  cache_hit_rate=0.9  avg margin=$18.10  loss-making users=1.45%

Routing-mix sensitivity (no caching change):
  cheap_routing_fraction=0.0  avg margin=$17.63  loss-making users=1.85%
  cheap_routing_fraction=0.3  avg margin=$18.10  loss-making users=1.40%
  cheap_routing_fraction=0.6  avg margin=$18.58  loss-making users=0.95%
  cheap_routing_fraction=0.9  avg margin=$19.05  loss-making users=0.40%


Routing mix moves the margin more than caching does *in this model* — because caching only discounts the input side, and **output tokens are priced 5x higher than input** for this tier, so a change that touches output cost (routing to a cheaper tier) has more leverage than a change that only touches input cost (caching). That's not a universal law — a workload dominated by a huge repeated context (a large RAG corpus, a long system prompt) would see caching matter far more — but it's exactly the kind of thing this sensitivity analysis is for: knowing *which* lever to pull for *your* workload's actual cost shape, instead of reaching for whichever one is fashionable.

## Part 2 — Building the router notebook 32 only sketched

Notebook 32's router made one binary decision (cheap vs. expensive) based on one heuristic (hedge words in the answer). A router that has to serve real declared constraints — a quality floor for a compliance-sensitive feature, a latency ceiling for an interactive one, a hard budget for a free tier — needs a real **scorecard** per model: measured latency, real cost, and a quality score from an actual eval, not a guess.


In [3]:
# Reusing notebook 24's exact harness shape: a golden dataset, a scorer, aggregate with
# np.mean -- NOT a new eval framework. Quality here is measured, not assumed.
GOLDEN = [
    {"q": "What is the capital of France?", "ref": "paris"},
    {"q": "What is the capital of Japan?", "ref": "tokyo"},
    {"q": "What is 12 + 7?", "ref": "19"},
    {"q": "What is the capital of Australia?", "ref": "canberra"},
    {"q": "What is 9 * 6?", "ref": "54"},
    {"q": "What is the capital of Canada?", "ref": "ottawa"},
    {"q": "What is 15 - 6?", "ref": "9"},
    {"q": "What is the capital of Germany?", "ref": "berlin"},
]

def exact_match_score(output, ref):
    return 1.0 if ref.lower() in output.lower() else 0.0

def simulate_tier_output(item, true_accuracy, rng):
    """Offline stand-in for a real model call: with probability true_accuracy, the
    tier answers correctly; otherwise it produces a plausible wrong (non-)answer. Swap
    this for a real client.messages.create(...) call -- see the guarded live cell below
    -- and the router mechanics below don't change at all.
    """
    return item["ref"] if rng.random() < true_accuracy else "unable to determine"

MODEL_TIERS = {
    "haiku":  {"input_price": 1.00, "output_price": 5.00,  "true_accuracy": 0.75, "base_latency_s": 0.4},
    "sonnet": {"input_price": 3.00, "output_price": 15.00, "true_accuracy": 0.90, "base_latency_s": 0.9},
    "opus":   {"input_price": 5.00, "output_price": 25.00, "true_accuracy": 0.97, "base_latency_s": 1.8},
}

class ModelRouter:
    """A per-model scorecard (quality, cost, measured latency) driving routing decisions."""
    def __init__(self, tiers):
        self.tiers = tiers
        self.measured_latency = {name: cfg["base_latency_s"] for name, cfg in tiers.items()}
        self.quality = {}

    def evaluate_quality(self, golden, scorer, rng):
        """notebook 24's run_variant -> scorer -> aggregate shape, one variant per tier."""
        for name, cfg in self.tiers.items():
            outputs = [simulate_tier_output(item, cfg["true_accuracy"], rng) for item in golden]
            scores = [scorer(o, item["ref"]) for o, item in zip(outputs, golden)]
            self.quality[name] = float(np.mean(scores))

    def observe_latency(self, name, latency_s, alpha=0.3):
        """Exponentially-weighted rolling average -- the same idea as 29c's Gauge metrics,
        just updated in Python instead of scraped from /metrics.
        """
        self.measured_latency[name] = alpha * latency_s + (1 - alpha) * self.measured_latency[name]

    def cost_per_request(self, name, in_tok, out_tok):
        cfg = self.tiers[name]
        return (in_tok / 1e6) * cfg["input_price"] + (out_tok / 1e6) * cfg["output_price"]

router = ModelRouter(MODEL_TIERS)
# A DEDICATED rng for the router's quality eval -- reusing Part 1's `rng` here would make
# these scores depend on how many random draws Part 1 happened to consume first (which is
# exactly the bug that produced three identical 1.000 scores while writing this notebook).
router_rng = np.random.default_rng(1)
router.evaluate_quality(GOLDEN, exact_match_score, router_rng)
for name in MODEL_TIERS:
    print(f"{name:7s}  quality={router.quality[name]:.3f}  "
          f"latency={router.measured_latency[name]:.2f}s  "
          f"cost/req=${router.cost_per_request(name, 800, 500):.5f}")


haiku    quality=0.625  latency=0.40s  cost/req=$0.00330
sonnet   quality=1.000  latency=0.90s  cost/req=$0.00990
opus     quality=0.875  latency=1.80s  cost/req=$0.01650


Look closely at those quality numbers: **sonnet (0.90 true accuracy) scored higher than opus (0.97 true accuracy)**. That's not a bug — it's the same small-golden-set noise notebook 24 warned about, showing up somewhere it actually costs money. With only `len(GOLDEN)` golden items, quality is quantized in steps of `1/len(GOLDEN)`; a couple of unlucky draws are enough to flip the ranking between two models whose true accuracies are 7 points apart. A router built on this scorecard would confidently pick sonnet over opus for any quality bar between 0.90 and 1.00 — the *wrong* call, made with total confidence, because the eval set was too small to tell the difference. The fix isn't a bigger rng seed, it's a bigger golden set (or repeated sampling) before trusting the ranking in production.

## Routing policies against declared constraints

A real router doesn't pick "the best model" in the abstract — it picks the model that satisfies *this request's* constraints most cheaply. (Keep the noise above in mind: these decisions are only as good as the scorecard behind them.)


In [4]:
def cheapest_above_quality_bar(router, quality_bar, in_tok, out_tok):
    candidates = [name for name in router.tiers if router.quality[name] >= quality_bar]
    if not candidates:
        return max(router.tiers, key=lambda n: router.quality[n])  # nothing clears the bar; use the best
    return min(candidates, key=lambda n: router.cost_per_request(n, in_tok, out_tok))

def cheapest_under_latency_ceiling(router, latency_ceiling_s, in_tok, out_tok):
    candidates = [name for name in router.tiers if router.measured_latency[name] <= latency_ceiling_s]
    if not candidates:
        return min(router.tiers, key=lambda n: router.measured_latency[n])  # nothing qualifies; use the fastest
    return min(candidates, key=lambda n: router.cost_per_request(n, in_tok, out_tok))

print("Quality bar 0.85 ->", cheapest_above_quality_bar(router, 0.85, 800, 500))
print("Quality bar 0.99 ->", cheapest_above_quality_bar(router, 0.99, 800, 500))
print("Latency ceiling 0.5s ->", cheapest_under_latency_ceiling(router, 0.5, 800, 500))
print("Latency ceiling 2.0s ->", cheapest_under_latency_ceiling(router, 2.0, 800, 500))


Quality bar 0.85 -> sonnet
Quality bar 0.99 -> sonnet
Latency ceiling 0.5s -> haiku
Latency ceiling 2.0s -> haiku


## Part 3 — The off-the-shelf version

Building a scorecard router from scratch teaches the mechanics; in production most teams reach for a gateway that already implements this. **LiteLLM** does cost-based and latency-based routing out of the box, over 100+ providers, through one OpenAI-compatible interface — and it treats a locally-running model exactly like a cloud one, which is where **LM Studio** and **Ollama** come in as backends.


In [5]:
try:
    import litellm
    from litellm import Router
    HAS_LITELLM = True
except ImportError:
    HAS_LITELLM = False

print(f"litellm installed: {HAS_LITELLM}")

if HAS_LITELLM:
    # Configuring the router does NOT make any network call -- this is purely the shape
    # a production LiteLLM gateway config takes, built from this notebook's own pricing.
    model_list = [
        {"model_name": "cheap-tier", "litellm_params": {
            "model": "claude-haiku-4-5",
            "input_cost_per_token": MODEL_TIERS["haiku"]["input_price"] / 1e6,
            "output_cost_per_token": MODEL_TIERS["haiku"]["output_price"] / 1e6,
        }},
        {"model_name": "balanced-tier", "litellm_params": {
            "model": "claude-sonnet-5",
            "input_cost_per_token": MODEL_TIERS["sonnet"]["input_price"] / 1e6,
            "output_cost_per_token": MODEL_TIERS["sonnet"]["output_price"] / 1e6,
        }},
    ]
    litellm_router = Router(model_list=model_list, routing_strategy="cost-based-routing")
    print(f"LiteLLM router configured with routing_strategy='cost-based-routing'")
    print(f"Model groups: {litellm_router.model_names}")
    print("A real call would be: litellm_router.completion(model='cheap-tier', messages=[...])")
    print("-- the SAME call shape works whether 'cheap-tier' points at a hosted API or a")
    print("local backend below; the router config is where the routing logic lives, not")
    print("the call site.")


litellm installed: True
LiteLLM router configured with routing_strategy='cost-based-routing'
Model groups: {'balanced-tier', 'cheap-tier'}
A real call would be: litellm_router.completion(model='cheap-tier', messages=[...])
-- the SAME call shape works whether 'cheap-tier' points at a hosted API or a
local backend below; the router config is where the routing logic lives, not
the call site.


In [6]:
# Guarded reachability checks -- LM Studio and Ollama are both just local OpenAI-compatible
# servers a router can treat as additional tiers, with $0 marginal cost per request.
def port_open(host, port, timeout=0.5):
    import socket
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

HAS_LM_STUDIO = port_open("127.0.0.1", 1234)
HAS_OLLAMA = port_open("127.0.0.1", 11434)
print(f"LM Studio reachable on :1234: {HAS_LM_STUDIO}")
print(f"Ollama reachable on :11434:   {HAS_OLLAMA}")

if not HAS_LM_STUDIO:
    print("\n[LM Studio not running] Start its local server (Developer tab -> Start Server),")
    print("then point any OpenAI-compatible client -- including litellm.completion(model=")
    print("'lm_studio/<loaded-model>', ...) -- at http://localhost:1234/v1. Zero token cost,")
    print("bounded only by local hardware.")
if not HAS_OLLAMA:
    print("\n[Ollama not running] `ollama serve` exposes the same OpenAI-compatible surface")
    print("on :11434 -- litellm.completion(model='ollama/llama3.1:8b', api_base=")
    print("'http://localhost:11434') routes to it with no code change beyond the model string.")


LM Studio reachable on :1234: False
Ollama reachable on :11434:   False

[LM Studio not running] Start its local server (Developer tab -> Start Server),
then point any OpenAI-compatible client -- including litellm.completion(model=
'lm_studio/<loaded-model>', ...) -- at http://localhost:1234/v1. Zero token cost,
bounded only by local hardware.

[Ollama not running] `ollama serve` exposes the same OpenAI-compatible surface
on :11434 -- litellm.completion(model='ollama/llama3.1:8b', api_base=
'http://localhost:11434') routes to it with no code change beyond the model string.


## Exercises


In [7]:
# Exercise 1 (Warm-up): How much does golden-set size change the quality estimate?
# Task: Re-run router.evaluate_quality with GOLDEN extended to 20 items (repeat the
#       pattern: simple facts and arithmetic with unambiguous answers) instead of 8. Do the
#       quality scores change? Are they less "quantized" (fewer round multiples of 0.125)?
# Hint: quality with N items can only take values that are multiples of 1/N -- more items
#       means finer-grained (and less noisy) quality estimates, at the cost of more calls
#       per evaluation.

# YOUR CODE HERE


In [8]:
# Exercise 2 (Apply): Implement budget-aware routing over a replayed request log
# Task: Write budget_aware_route(router, remaining_budget, in_tok, out_tok, quality_bar=0.85)
#       that behaves like cheapest_above_quality_bar UNLESS remaining_budget is too low to
#       afford even the CHEAPEST tier meeting the bar for this request -- in that case, drop
#       the quality bar requirement and return the single cheapest tier available (serving a
#       lower-quality answer beats a hard failure). Then replay a log of 200 requests (build
#       one: random in_tok/out_tok per request, e.g. rng.integers(200,1200) and
#       rng.integers(100,800)) against a starting budget of $0.50, decrementing the budget by
#       the ACTUAL cost of whichever tier got chosen each time. Print how many requests got
#       downgraded once the budget ran low, and at what request index it first happened.
# Hint: candidates = [n for n in router.tiers if router.quality[n] >= quality_bar]; if the
#       cheapest candidate's cost > remaining_budget, fall back to
#       min(router.tiers, key=lambda n: router.cost_per_request(n, in_tok, out_tok)).

# YOUR CODE HERE


In [9]:
# Exercise 3 (Extend): Connect unit economics to the router
# Task: Using population_margin(cheap_fraction=X), find the SMALLEST cheap_fraction (search
#       in steps of 0.05) that gets the fraction of loss-making users (cost > $20) below
#       0.5%. Then connect this number back to the router: cheapest_above_quality_bar
#       effectively chooses a per-REQUEST cheap_fraction based on the quality bar you set --
#       write one sentence on how you'd pick a quality bar for a real feature so that the
#       AGGREGATE cheap_fraction across a real request mix lands near the number you just
#       found, instead of picking a quality bar arbitrarily.

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
GOLDEN_20 = GOLDEN + [
    {"q": "What is the capital of Italy?", "ref": "rome"},
    {"q": "What is the capital of Spain?", "ref": "madrid"},
    {"q": "What is 8 * 8?", "ref": "64"},
    {"q": "What is the capital of Egypt?", "ref": "cairo"},
    {"q": "What is 100 - 37?", "ref": "63"},
    {"q": "What is the capital of Brazil?", "ref": "brasilia"},
    {"q": "What is 6 * 7?", "ref": "42"},
    {"q": "What is the capital of India?", "ref": "delhi"},
    {"q": "What is 45 + 55?", "ref": "100"},
    {"q": "What is the capital of Russia?", "ref": "moscow"},
    {"q": "What is 13 * 4?", "ref": "52"},
    {"q": "What is the capital of China?", "ref": "beijing"},
]
router_20 = ModelRouter(MODEL_TIERS)
router_20.evaluate_quality(GOLDEN_20, exact_match_score, router_rng)
for name in MODEL_TIERS:
    print(f"{name:7s}  quality (n=8): {router.quality[name]:.3f}   quality (n=20): {router_20.quality[name]:.3f}")
# With 20 items, quality can land on multiples of 0.05 instead of 0.125 -- a finer-grained,
# less noisy estimate of each tier's true accuracy, at 2.5x the evaluation cost.

# Exercise 2
def budget_aware_route(router, remaining_budget, in_tok, out_tok, quality_bar=0.85):
    candidates = [n for n in router.tiers if router.quality[n] >= quality_bar]
    if candidates:
        cheapest = min(candidates, key=lambda n: router.cost_per_request(n, in_tok, out_tok))
        if router.cost_per_request(cheapest, in_tok, out_tok) <= remaining_budget:
            return cheapest, False  # False = not downgraded
    # Either nothing meets the quality bar affordably, or nothing meets it at all --
    # fall back to the single cheapest tier, full stop.
    return min(router.tiers, key=lambda n: router.cost_per_request(n, in_tok, out_tok)), True

budget = 0.50
downgrades = 0
first_downgrade_at = None
for i in range(200):
    in_tok_i = int(rng.integers(200, 1200))
    out_tok_i = int(rng.integers(100, 800))
    tier, downgraded = budget_aware_route(router, budget, in_tok_i, out_tok_i)
    budget -= router.cost_per_request(tier, in_tok_i, out_tok_i)
    if downgraded:
        downgrades += 1
        if first_downgrade_at is None:
            first_downgrade_at = i
print(f"Downgraded {downgrades}/200 requests once budget ran low; first at request #{first_downgrade_at}")
# Early requests get the quality-bar-satisfying tier freely; once the running budget can no
# longer afford it, routing degrades gracefully to "cheapest available" instead of failing
# outright -- the same principle as notebook 29's fallback path, applied to cost instead of
# availability.

# Exercise 3
for cheap in np.arange(0.0, 1.01, 0.05):
    _, neg_frac = population_margin(cheap_fraction=cheap)
    if neg_frac < 0.005:
        print(f"Smallest cheap_fraction keeping loss-making users under 0.5%: {cheap:.2f}")
        break
# Picking a quality bar isn't just an accuracy decision -- it implicitly sets what fraction
# of real traffic routes to the cheap tier. The right process: measure what fraction of your
# ACTUAL request mix clears a candidate quality bar (via cheapest_above_quality_bar on
# representative requests, not golden-set requests), compare that fraction to the
# cheap_fraction this sensitivity analysis says you need, and adjust the bar until the two
# line up -- rather than setting a quality bar first and hoping the economics work out.
```
</details>


## Key Takeaways
- Unit economics for a flat-price AI feature lives in the **distribution** of cost per user, not the average — a healthy-looking mean can coexist with a real, growing loss-making tail, exactly the users a successful product creates more of.
- Which lever (caching vs. routing) improves margin more **depends on your workload's cost shape** — output-heavy workloads respond more to routing since output tokens are typically priced several times higher than input.
- A real router needs a **scorecard** — measured latency, real cost, and quality from an actual eval (reusing notebook 24's harness, not inventing a new one) — routing against *declared constraints* per request rather than one global heuristic.
- **LiteLLM** provides cost-based and latency-based routing over 100+ providers through one interface; **LM Studio** and **Ollama** slot into that same router config as zero-marginal-cost local tiers.
- Budget-aware routing degrades gracefully (serve a cheaper answer) rather than failing outright when a budget runs low — the cost-side analog of notebook 29's availability fallback.

## What's Next
Notebook **09_frontier/37 — Edge Deployment** pushes routing to its logical extreme: instead of choosing which *server* handles a request, it asks whether the request needs a server at all — running inference directly on a laptop, a phone, or inside a browser tab.
